In [1]:
import re
import os
import copy
import json
import random
import pickle
import logging
from pathlib import Path
from datetime import datetime
import networkx as nx
import matplotlib.pyplot as plt

In [2]:
ruta_archivo_norequ = r"C:\Users\alhel\Desktop\TESIS CARP 2025\Instancias\kshs\kshs1.dat" #this is the most general case

# Read instance .dat file based on format described in https://www.uv.es/~belengue/carp/READ_ME

In [2]:
import re

def leer_carplib_dat(filepath):
    parsed_data = {}
    current_list_key = None
    contador_tarea = 1
    
    with open(filepath, 'r', encoding='utf-8') as file:
        for line in file:
            line = line.strip()
            if not line or line.startswith(('=', '-', '|')):
                continue
            
            if ':' in line:
                key, val = line.split(':', 1)
                key = key.strip()
                val = val.strip()
                
                if key.startswith('LISTA_ARISTAS'):
                    current_list_key = key
                    parsed_data[key] = []
                    contador_tarea = 1
                else:
                    current_list_key = None
                    if val.isdigit() or (val.startswith('-') and val[1:].isdigit()):
                        parsed_data[key] = int(val)
                    else:
                        parsed_data[key] = val
                        
            elif current_list_key:
                numeros = [int(x) for x in re.findall(r'-?\d+', line)]
                if len(numeros) >= 3:
                    # Asignamos el prefijo dependiendo de la lista que se esté leyendo
                    if "NOREQ" in current_list_key:
                        prefijo = "TNR"
                    else:
                        prefijo = "TR"
                        
                    parsed_data[current_list_key].append({
                        'tarea': f"{prefijo}{contador_tarea}",
                        'nodos': (numeros[0], numeros[1]),
                        'costo': numeros[2],
                        'demanda': numeros[3] if len(numeros) >= 4 else 0
                    })
                    contador_tarea += 1

    nombre_objeto = 'loaded_dict'  
    return parsed_data, nombre_objeto

In [52]:
d_noreq, d_noreq_sname = leer_carplib_dat(ruta_archivo_norequ)
d_noreq, d_noreq_sname

({'NOMBRE': 'kshs1',
  'COMENTARIO': '69640.000000 (cota superior)',
  'VERTICES': 8,
  'ARISTAS_REQ': 15,
  'ARISTAS_NOREQ': 0,
  'VEHICULOS': 4,
  'CAPACIDAD': 150,
  'TIPO_COSTES_ARISTAS': 'EXPLICITOS',
  'COSTE_TOTAL_REQ': 8705,
  'LISTA_ARISTAS_REQ': [{'tarea': 'TR1',
    'nodos': (1, 3),
    'costo': 737,
    'demanda': 40},
   {'tarea': 'TR2', 'nodos': (1, 8), 'costo': 1082, 'demanda': 27},
   {'tarea': 'TR3', 'nodos': (2, 3), 'costo': 510, 'demanda': 25},
   {'tarea': 'TR4', 'nodos': (2, 4), 'costo': 166, 'demanda': 54},
   {'tarea': 'TR5', 'nodos': (2, 5), 'costo': 745, 'demanda': 5},
   {'tarea': 'TR6', 'nodos': (2, 6), 'costo': 780, 'demanda': 60},
   {'tarea': 'TR7', 'nodos': (2, 7), 'costo': 370, 'demanda': 30},
   {'tarea': 'TR8', 'nodos': (3, 4), 'costo': 663, 'demanda': 7},
   {'tarea': 'TR9', 'nodos': (3, 7), 'costo': 311, 'demanda': 50},
   {'tarea': 'TR10', 'nodos': (3, 8), 'costo': 696, 'demanda': 23},
   {'tarea': 'TR11', 'nodos': (4, 7), 'costo': 468, 'demanda': 3

In [53]:
d_noreq.values()

dict_values(['kshs1', '69640.000000 (cota superior)', 8, 15, 0, 4, 150, 'EXPLICITOS', 8705, [{'tarea': 'TR1', 'nodos': (1, 3), 'costo': 737, 'demanda': 40}, {'tarea': 'TR2', 'nodos': (1, 8), 'costo': 1082, 'demanda': 27}, {'tarea': 'TR3', 'nodos': (2, 3), 'costo': 510, 'demanda': 25}, {'tarea': 'TR4', 'nodos': (2, 4), 'costo': 166, 'demanda': 54}, {'tarea': 'TR5', 'nodos': (2, 5), 'costo': 745, 'demanda': 5}, {'tarea': 'TR6', 'nodos': (2, 6), 'costo': 780, 'demanda': 60}, {'tarea': 'TR7', 'nodos': (2, 7), 'costo': 370, 'demanda': 30}, {'tarea': 'TR8', 'nodos': (3, 4), 'costo': 663, 'demanda': 7}, {'tarea': 'TR9', 'nodos': (3, 7), 'costo': 311, 'demanda': 50}, {'tarea': 'TR10', 'nodos': (3, 8), 'costo': 696, 'demanda': 23}, {'tarea': 'TR11', 'nodos': (4, 7), 'costo': 468, 'demanda': 39}, {'tarea': 'TR12', 'nodos': (4, 8), 'costo': 1046, 'demanda': 55}, {'tarea': 'TR13', 'nodos': (5, 8), 'costo': 239, 'demanda': 29}, {'tarea': 'TR14', 'nodos': (6, 7), 'costo': 490, 'demanda': 65}, {'tare

## CHECKING THAT THE NUMBER OF LINES IS THE SAME AS THE NUMBER OF ITEMS IN DICT

In [3]:
def validate_instance(created_dict, dat_file_path): ### las no requeridas deben tener otro id, no Ti
    """
    Valida la integridad de la instancia comparando metadatos, 
    la longitud de las listas y el total de líneas útiles en el archivo original.
    """
    
    # 1. Extraemos los valores esperados (lo que dice el encabezado)
    n_req_esperadas = created_dict.get("ARISTAS_REQ", 0)
    n_noreq_esperadas = created_dict.get("ARISTAS_NOREQ", 0)
    
    # 2. Contamos cuánto se leyó realmente en las listas
    lista_req = created_dict.get("LISTA_ARISTAS_REQ", [])
    lista_noreq = created_dict.get("LISTA_ARISTAS_NOREQ", [])
    n_req_reales = len(lista_req)
    n_noreq_reales = len(lista_noreq)
    
    # 3. Validamos integridad de nodos (el nodo más alto no debe superar VERTICES)
    todas_las_aristas = lista_req + lista_noreq
    
    nodos_encontrados = []
    for item in todas_las_aristas:
        # Corregido: usamos 'nodos' en lugar de 'arco' basado en nuestra función anterior
        nodos_encontrados.extend(item['nodos'])
    
    max_nodo = max(nodos_encontrados) if nodos_encontrados else 0
    total_vertices = created_dict.get("VERTICES", 0)

    # 4. NUEVA LÓGICA: Contar líneas útiles en el archivo .dat
    lineas_utiles_archivo = 0
    with open(dat_file_path, 'r', encoding='utf-8') as file:
        for line in file:
            line = line.strip()
            # Aplicamos el mismo filtro que en la función de lectura
            if not line or line.startswith('=') or line.startswith('-') or line.startswith('|'):
                continue
            lineas_utiles_archivo += 1

    # El total de elementos capturados es: llaves principales + items en las listas
    elementos_en_dict = len(created_dict) + n_req_reales + n_noreq_reales

    # 5. Verificación de condiciones
    check_req = (n_req_esperadas == n_req_reales)
    check_noreq = (n_noreq_esperadas == n_noreq_reales)
    check_nodes = (max_nodo <= total_vertices)
    check_lineas = (lineas_utiles_archivo == elementos_en_dict)

    is_valid = check_req and check_noreq and check_nodes and check_lineas

    # 6. Resultado detallado
    validation_results = {
        "is_valid": is_valid,
        "status": "VALIDACIÓN EXITOSA" if is_valid else "ERROR DE INTEGRIDAD",
        "detalles": {
            "aristas_req": f"{n_req_reales}/{n_req_esperadas}",
            "aristas_noreq": f"{n_noreq_reales}/{n_noreq_esperadas}",
            "nodos_ok": f"Max Nodo {max_nodo} <= {total_vertices}",
            "lineas_vs_dict": f"Archivo: {lineas_utiles_archivo} == Dict: {elementos_en_dict}"
        }
    }

    nombre_objeto = 'validation_dict'
    return validation_results, nombre_objeto

In [55]:
reading_validation_dict_noreq, val_d_name = validate_instance(d_noreq, ruta_archivo_norequ)
reading_validation_dict_noreq, val_d_name

({'is_valid': True,
  'status': 'VALIDACIÓN EXITOSA',
  'detalles': {'aristas_req': '15/15',
   'aristas_noreq': '0/0',
   'nodos_ok': 'Max Nodo 8 <= 8',
   'lineas_vs_dict': 'Archivo: 26 == Dict: 26'}},
 'validation_dict')

# GESTIÓN DE EJECUCIÓN Y GUARDADO

In [4]:
def iniciar_ejecucion(metadata_dict, base_dir="corridas_files"):
    project_name = metadata_dict.get("NOMBRE", "Instancia")
    run_id = datetime.now().strftime("%H%M%S")
    date_str = datetime.now().strftime("%Y-%m-%d")

    try:
        directorio_base = Path(__file__).resolve().parent
    except NameError:
        directorio_base = Path.cwd()
        
    run_path = directorio_base / base_dir / f"{project_name}_{date_str}__ID-{run_id}"
    run_path.mkdir(parents=True, exist_ok=True)

    logger = logging.getLogger(f"{project_name}_{run_id}")
    logger.setLevel(logging.INFO)
    
    if not logger.handlers:
        file_handler = logging.FileHandler(run_path / "execution.log")
        file_handler.setFormatter(logging.Formatter('%(asctime)s - %(message)s'))
        logger.addHandler(file_handler)

    logger.info("Run initialized.")
    return run_path, logger

In [57]:
tst_path, tst_logger = iniciar_ejecucion(d_noreq, base_dir="instance_runs_test_debug_20260313")

In [58]:
tst_path

WindowsPath('C:/Users/alhel/Desktop/TESIS CARP 2025/Metaheuristicas/TestingClass/instance_runs_test_debug_20260313/kshs1_2026-03-18__ID-003344')

In [5]:
import os
import json
import pickle

def guardar_objeto_automatico(carpeta, nombre_instancia, objeto, tipo_objeto):
    """
    Guarda un objeto en disco infiriendo su formato ideal.
    El archivo resultante tendrá el formato: carpeta/nombre_instancia_tipo_objeto.ext
    """
    # Construimos el nombre base estandarizado
    nombre_base = f"{nombre_instancia}_{tipo_objeto}"
    
    # 1. Gráficas o figuras (Matplotlib/Plotly compatibles)
    if hasattr(objeto, 'savefig'):
        ruta = os.path.join(carpeta, f"{nombre_base}.jpg")
        objeto.savefig(ruta, format='jpg', dpi=300, bbox_inches='tight')
        return ruta
        
    # 2. Cadenas de texto plano (Logs, reportes)
    elif isinstance(objeto, str):
        ruta = os.path.join(carpeta, f"{nombre_base}.txt")
        with open(ruta, 'w', encoding='utf-8') as f:
            f.write(objeto)
        return ruta
        
    # 3. Diccionarios o Listas (Intentamos guardar como JSON para legibilidad)
    elif isinstance(objeto, (dict, list)):
        ruta_json = os.path.join(carpeta, f"{nombre_base}.json")
        try:
            with open(ruta_json, 'w', encoding='utf-8') as f:
                json.dump(objeto, f, indent=4)
            return ruta_json
        except TypeError:
            # Si el diccionario tiene objetos no serializables en JSON (como tuplas como llaves), 
            # fallará silenciosamente y pasará al guardado en Pickle.
            pass 
            
    # 4. Fallback de seguridad (Pickle binario para cualquier objeto complejo)
    ruta_pkl = os.path.join(carpeta, f"{nombre_base}.pkl")
    with open(ruta_pkl, 'wb') as f:
        pickle.dump(objeto, f)
        
    return ruta_pkl

In [60]:
guardar_objeto_automatico(tst_path, d_noreq['NOMBRE'],d_noreq, d_noreq_sname)

'C:\\Users\\alhel\\Desktop\\TESIS CARP 2025\\Metaheuristicas\\TestingClass\\instance_runs_test_debug_20260313\\kshs1_2026-03-18__ID-003344\\kshs1_loaded_dict.json'

In [61]:
guardar_objeto_automatico(tst_path, d_noreq['NOMBRE'],reading_validation_dict_noreq, val_d_name)

'C:\\Users\\alhel\\Desktop\\TESIS CARP 2025\\Metaheuristicas\\TestingClass\\instance_runs_test_debug_20260313\\kshs1_2026-03-18__ID-003344\\kshs1_validation_dict.json'

In [6]:
def finalizar_ejecucion(run_path, logger, success=True, reason=""):
    status_str = "SUCCESS" if success else "FAILED"
    (run_path / f"STATUS_{status_str}").touch()
    
    mensaje = f"Run finished with status: {status_str}"
    if reason: 
        mensaje += f" | Reason: {reason}"
        
    logger.info(mensaje)

# Draw the graph based on the arcs and nodes extracted in the previous step

In [7]:
def generar_grafo_limpio(data, carpeta_salida=None, figsize=(35,30), k_layout=15):
    G = nx.Graph()
    for i in range(1, data.get('VERTICES', 0) + 1):
        G.add_node(i)
        
    for item in data.get('LISTA_ARISTAS_REQ', []):
        u, v = item['nodos']
        G.add_edge(u, v, cost=item['costo'], demand=item['demanda'], tipo='req')

    for item in data.get('LISTA_ARISTAS_NOREQ', []):
        u, v = item['nodos']
        G.add_edge(u, v, cost=item['costo'], demand=item.get('demanda', 0), tipo='noreq')

    pos = nx.spring_layout(G, k=k_layout, iterations=200, seed=42)
    plt.figure(figsize=figsize) 
    
    deposito = data.get('DEPOSITO', 1)
    node_colors = ['#ff4d4d' if node == deposito else '#74b9ff' for node in G.nodes()]
    
    nx.draw_networkx_nodes(G, pos, node_size=2000, node_color=node_colors, edgecolors='#74b9ff', linewidths=1)
    nx.draw_networkx_labels(G, pos, font_size=25, font_weight='bold')
    
    arcos_req = [(u, v) for u, v, d in G.edges(data=True) if d['tipo'] == 'req']
    arcos_noreq = [(u, v) for u, v, d in G.edges(data=True) if d['tipo'] == 'noreq']
    
    nx.draw_networkx_edges(G, pos, edgelist=arcos_req, width=2.0, edge_color='black')
    nx.draw_networkx_edges(G, pos, edgelist=arcos_noreq, width=1.5, edge_color='gray', style='dashed')
    
    edge_labels = {}
    for u, v, d in G.edges(data=True):
        if d['tipo'] == 'req':
            edge_labels[(u, v)] = f"C:{d['cost']}\nD:{d['demand']}"
        else:
            edge_labels[(u, v)] = f"C:{d['cost']}"
            
    nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_size=15, font_color='#2d3436', label_pos=0.5, bbox=dict(alpha=0))
    
    texto_resumen = (
        f"RESUMEN DE LA INSTANCIA\n"
        f"--------------------------------------\n"
        f"Nombre: {data.get('NOMBRE', 'N/A')}\n"
        f"Vértices: {data.get('VERTICES', 0)}\n"
        f"Vehículos: {data.get('VEHICULOS', 0)}\n"
        f"Capacidad: {data.get('CAPACIDAD', 0)}\n"
        f"Aristas Req: {data.get('ARISTAS_REQ', 0)}\n"
        f"Aristas No Req: {data.get('ARISTAS_NOREQ', 0)}"
    )
    
    plt.text(0.02, 0.98, texto_resumen, transform=plt.gca().transAxes, fontsize=20, 
             verticalalignment='top', bbox=dict(boxstyle='round,pad=0.5', facecolor='#f1f2f6', alpha=0.9))

    nombre_instancia = data.get('NOMBRE', 'instancia')
    plt.title(f"Grafo de Instancia: {nombre_instancia}\n"
              f"(Rojo = Depósito | Negro Sólido = Req | Gris Punteado = No Req)", 
              fontsize=25, pad=20)
    plt.axis('off')
    plt.tight_layout()

    # NUEVO: Guardar la imagen de Matplotlib directamente aquí
    # (Asumiendo que le pasas 'carpeta_salida' a la función)
    if carpeta_salida in locals():
        ruta_imagen = os.path.join(carpeta_salida, f"{nombre_instancia}_mapa_estatico.png")
        plt.savefig(ruta_imagen, format='png', dpi=300, bbox_inches='tight')
    else:
        pass
  #  plt.show()
    plt.close() # Cierra la figura visual
    
    #object_name = "grafo_matematico"
    return G

In [67]:
G_test = generar_grafo_limpio(d_noreq, carpeta_salida = None, figsize=(35,30), k_layout=15)
#G_test

# Generar la distancia mas corta (costo) entre cada par de nodos

In [8]:
import networkx as nx
import matplotlib.pyplot as plt

In [9]:
import networkx as nx

def calcular_matriz_distancias(G, algoritmo='dijkstra', carpeta_salida=None, nombre_instancia="instancia"):
    algoritmo = algoritmo.strip().lower()
    nodos = list(G.nodes())
    matriz_distancias = {}
    
    if algoritmo == 'dijkstra':
        dijkstra_raw = dict(nx.all_pairs_dijkstra_path_length(G, weight='cost'))
        for u in nodos:
            matriz_distancias[u] = {}
            for v in nodos:
                matriz_distancias[u][v] = dijkstra_raw.get(u, {}).get(v, float('inf'))
                
    elif algoritmo in ['floyd-warshall', 'floyd_warshall', 'floyd']:
        fw_raw = nx.floyd_warshall(G, weight='cost')
        for u in nodos:
            matriz_distancias[u] = dict(fw_raw[u])
            
    # Añadido para hacer match con el selector de la interfaz gráfica (app.py)
    elif algoritmo in ['bellman-ford', 'bellman_ford', 'bellman']:
        bf_raw = dict(nx.all_pairs_bellman_ford_path_length(G, weight='cost'))
        for u in nodos:
            matriz_distancias[u] = {}
            for v in nodos:
                matriz_distancias[u][v] = bf_raw.get(u, {}).get(v, float('inf'))
                
    else:
        raise ValueError(f"Algoritmo '{algoritmo}' no reconocido.")
        
    # NUEVO: Guardado Automático en disco
    if carpeta_salida:
        # El nombre quedará como: nombre_instancia_matriz_distancias_dijkstra.json
        tipo_objeto = f"matriz_distancias_{algoritmo}"
        # Se asume que guardar_objeto_automatico está definida en este mismo archivo
        guardar_objeto_automatico(carpeta_salida, nombre_instancia,  matriz_distancias, tipo_objeto)
        
    return matriz_distancias

In [72]:
matriz_distancias_t = calcular_matriz_distancias(G_test, algoritmo='dijkstra', carpeta_salida=None, nombre_instancia = d_noreq['NOMBRE'])
matriz_distancias_t

{1: {1: 0, 2: 1247, 3: 737, 4: 1400, 5: 1321, 6: 1484, 7: 1048, 8: 1082},
 2: {1: 1247, 2: 0, 3: 510, 4: 166, 5: 745, 6: 780, 7: 370, 8: 984},
 3: {1: 737, 2: 510, 3: 0, 4: 663, 5: 935, 6: 801, 7: 311, 8: 696},
 4: {1: 1400, 2: 166, 3: 663, 4: 0, 5: 911, 6: 946, 7: 468, 8: 1046},
 5: {1: 1321, 2: 745, 3: 935, 4: 911, 5: 0, 6: 641, 7: 1115, 8: 239},
 6: {1: 1484, 2: 780, 3: 801, 4: 946, 5: 641, 6: 0, 7: 490, 8: 402},
 7: {1: 1048, 2: 370, 3: 311, 4: 468, 5: 1115, 6: 490, 7: 0, 8: 892},
 8: {1: 1082, 2: 984, 3: 696, 4: 1046, 5: 239, 6: 402, 7: 892, 8: 0}}

# Factibilidad

In [10]:
import random

def es_ruta_factible(ruta, data, matriz_distancias):
    """Verifica si una sola ruta cumple con capacidad y conectividad."""
    if not ruta: return True
    
    capacidad_max = data.get('CAPACIDAD', 0)
    deposito = data.get('DEPOSITO', 1)
    info_tareas = {t['tarea']: t for t in data.get('LISTA_ARISTAS_REQ', [])}
    
    demanda_total = 0
    nodos_anteriores = (deposito, deposito)
    
    for id_tarea in ruta:
        tarea = info_tareas.get(id_tarea)
        if not tarea: return False
        
        u_act, v_act = tarea['nodos']
        demanda_total += tarea['demanda']
        
        # 1. Chequeo de Capacidad
        if demanda_total > capacidad_max: 
            return False
            
        # 2. Chequeo de Conectividad (¿Podemos llegar desde la tarea anterior a esta?)
        u_ant, v_ant = nodos_anteriores
        hay_camino = (
            matriz_distancias[u_ant][u_act] != float('inf') or 
            matriz_distancias[u_ant][v_act] != float('inf') or
            matriz_distancias[v_ant][u_act] != float('inf') or 
            matriz_distancias[v_ant][v_act] != float('inf')
        )
        
        if not hay_camino: return False
        nodos_anteriores = (u_act, v_act)
        
    # 3. Retorno al depósito (¿Podemos volver a casa desde la última tarea?)
    u_ant, v_ant = nodos_anteriores
    if matriz_distancias[u_ant][deposito] == float('inf') and matriz_distancias[v_ant][deposito] == float('inf'):
        return False
        
    return True

In [11]:
def es_solucion_factible(solucion, data, matriz_distancias):
    """Verifica si toda la solución (flota de vehículos) es válida."""
    num_vehiculos = data.get('VEHICULOS', 0)
    
    # 1. Checar que no se utilicen más camiones de los permitidos
    rutas_activas = [r for r in solucion if r]
    if len(rutas_activas) > num_vehiculos:
        return False
        
    # 2. Validar cada ruta individualmente
    for ruta in solucion:
        if not es_ruta_factible(ruta, data, matriz_distancias):
            return False
            
    return True

# SOLUCION ALEATORIA

In [ ]:
import random

def generar_solucion_inicial_aleatoria(data, matriz_distancias, max_intentos=None, carpeta_salida=None, nombre_instancia="instancia"):
    """
    Construye aleatoriamente una solución. 
    Busca infinitamente (o hasta encontrarla) una solución inicial factible.
    """
    num_vehiculos = data.get('VEHICULOS', 0)
    capacidad_max = data.get('CAPACIDAD', 0)
    tareas_requeridas = data.get('LISTA_ARISTAS_REQ', [])
    
    # Chequeo de viabilidad física (Pre-requisito absoluto)
    for t in tareas_requeridas:
        if t['demanda'] > capacidad_max: 
            arco = t.get('nodos', 'Desconocido')
            raise ValueError(f"Instancia inviable: La tarea {t['tarea']} (Arco: {arco}) excede la capacidad {capacidad_max}.")

    intentos = 0
    
    # Bucle "infinito" hasta encontrar factibilidad
    while True:
        intentos += 1
        
        # Mezclamos las tareas para intentar un agrupamiento distinto
        tareas_mezcladas = tareas_requeridas.copy()
        random.shuffle(tareas_mezcladas)
        
        solucion = [[] for _ in range(num_vehiculos)]
        v_idx = 0
        demanda_actual = 0
        fallo_agrupamiento = False
        
        # 1. Construcción voraz aleatorizada
        for tarea in tareas_mezcladas:
            # Si cabe en el vehículo actual
            if demanda_actual + tarea['demanda'] <= capacidad_max:
                solucion[v_idx].append(tarea['tarea'])
                demanda_actual += tarea['demanda']
            else:
                # Intentamos pasar al siguiente vehículo
                v_idx += 1
                if v_idx >= num_vehiculos:
                    # Si no hay más vehículos, la solución es infactible
                    fallo_agrupamiento = True
                    break 
                
                # Asignamos al nuevo vehículo
                solucion[v_idx].append(tarea['tarea'])
                demanda_actual = tarea['demanda']
        
        # Si logramos meter todas las tareas en los vehículos disponibles...
        if not fallo_agrupamiento:
            # 2. Validación experta (Rutas, conectividad, etc.)
            if es_solucion_factible(solucion, data, matriz_distancias):
                if carpeta_salida:
                    guardar_objeto_automatico(carpeta_salida, nombre_instancia, solucion, "initial_random_solution")
                
                # Log opcional para ver cuánta resistencia puso la instancia
                if intentos > 1000:
                    print(f"✅ Solución inicial para {nombre_instancia} hallada tras {intentos} intentos.")
                
                return solucion

        # Feedback visual para procesos largos (cada 10,000 intentos)
        if intentos % 10000 == 0:
            print(f"⏳ Buscando inicio factible para {nombre_instancia}... ({intentos} intentos)")

In [147]:
random_sol_t = generar_solucion_inicial_aleatoria(d_noreq, matriz_distancias_t, max_intentos=10000,
                                                  carpeta_salida=tst_path, nombre_instancia=d_noreq['NOMBRE'])
random_sol_t

[['TR6', 'TR1', 'TR7'],
 ['TR15', 'TR14', 'TR12'],
 ['TR3', 'TR4', 'TR10', 'TR11'],
 ['TR13', 'TR9', 'TR8', 'TR2', 'TR5']]

In [148]:
es_solucion_factible(random_sol_t, d_noreq, matriz_distancias_t)

True

In [13]:
def detalle_sol(solucion, data, G, carpeta_salida=None, nombre_instancia="instancia" ):
    
    deposito = data.get('DEPOSITO', 1)
    capacidad_max = data.get('CAPACIDAD', 0)
    info_tareas = {t['tarea']: {'u': t['nodos'][0], 'v': t['nodos'][1], 'costo': t['costo'], 'demanda': t['demanda']}
                   for t in data.get('LISTA_ARISTAS_REQ', [])}
        
    costos_rutas = []
    costo_total_solucion = 0
    reporte = [] # Guardaremos el reporte en una lista para retornarlo como string
    
    reporte.append("="*80)
    reporte.append("EVALUACIÓN COMPACTA DE RUTAS")
    reporte.append("="*80)
    
    for i, ruta in enumerate(solucion):
        reporte.append(f"RUTA {i + 1} {ruta}")
        if not ruta:
            reporte.append(f"  -> Vehículo vacío | Costo Total: 0 | Demanda: 0 / {capacidad_max}\n")
            costos_rutas.append(0)
            continue
            
        costo_vehiculo, demanda_vehiculo, nodo_actual = 0, 0, deposito
        for id_tarea in ruta:
            tarea = info_tareas[id_tarea]
            u, v, costo_serv, dem_serv = tarea['u'], tarea['v'], tarea['costo'], tarea['demanda']
            
            if nodo_actual != u:
                camino_dh = nx.shortest_path(G, source=nodo_actual, target=u, weight='cost')
                costo_dh = nx.shortest_path_length(G, source=nodo_actual, target=u, weight='cost')
                str_dh = " -> ".join(map(str, camino_dh))
            else:
                costo_dh, str_dh = 0, f"Ninguno (ya en {u})"
                
            costo_total_paso = costo_dh + costo_serv
            costo_vehiculo += costo_total_paso
            demanda_vehiculo += dem_serv
            
            reporte.append(f"  -> {id_tarea} ({u},{v}) -> DH: [{str_dh}] | Demanda: {dem_serv} | Costo (DH + Serv): {costo_dh} + {costo_serv} = {costo_total_paso}")
            nodo_actual = v
            
        if nodo_actual != deposito:
            camino_ret = nx.shortest_path(G, source=nodo_actual, target=deposito, weight='cost')
            costo_ret = nx.shortest_path_length(G, source=nodo_actual, target=deposito, weight='cost')
            str_ret = " -> ".join(map(str, camino_ret))
        else:
            costo_ret, str_ret = 0, f"Ninguno (ya en {deposito})"
            
        costo_vehiculo += costo_ret
        reporte.append(f"  -> REGRESO A DEPÓSITO ({deposito}) -> DH: [{str_ret}] | Costo Regreso: {costo_ret}")
        
        estado_cap = "OK" if demanda_vehiculo <= capacidad_max else "EXCEDIDA"
        reporte.append(f"  => TOTAL RUTA {i + 1}: Costo Total = {costo_vehiculo} | Demanda Total = {demanda_vehiculo} / {capacidad_max} [{estado_cap}]\n")
        costos_rutas.append(costo_vehiculo)
        costo_total_solucion += costo_vehiculo
        
    reporte.append("="*80)
    reporte.append(f"COSTO TOTAL DE LA SOLUCIÓN: {costo_total_solucion}")
    reporte.append("="*80 + "\n")
    
    # Imprimimos y también retornamos el texto para poder guardarlo
    texto_final = "\n".join(reporte)

    if carpeta_salida:
       guardar_objeto_automatico(carpeta_salida, nombre_instancia, texto_final, "initial_random_solution_detail")

    return costos_rutas, costo_total_solucion, texto_final

In [150]:
c_, cts_, tf = detalle_sol(random_sol_t, d_noreq, G_test, carpeta_salida=tst_path, nombre_instancia=d_noreq['NOMBRE'])

In [14]:

import random
import math
import time

# ==========================================
# OPERADORES EVOLUTIVOS MEJORADOS (VARIEDAD)
# ==========================================

def op_inter_2opt(r1, r2):
    """2-Opt Inter-ruta: Cruza las colas de dos rutas para mejorar la topología."""
    h1, h2 = r1.copy(), r2.copy()
    if len(h1) < 2 or len(h2) < 2: # Evitamos errores en rutas casi vacías
        return h1, h2, {'info': 'Rutas muy cortas'}
    c1 = random.randint(1, len(h1) - 1) # Punto de corte aleatorio ruta 1
    c2 = random.randint(1, len(h2) - 1) # Punto de corte aleatorio ruta 2
    nueva_r1 = h1[:c1] + h2[c2:] # Reconexión A: inicio R1 + final R2
    nueva_r2 = h2[:c2] + h1[c1:] # Reconexión B: inicio R2 + final R1
    return nueva_r1, nueva_r2, {'op': '2opt_inter'}

def op_inter_swap(r1, r2):
    """Intercambio 1 a 1: Crucial para mantener la capacidad en instancias pequeñas."""
    h1, h2 = r1.copy(), r2.copy()
    if not h1 or not h2: return h1, h2, {'info': 'vacío'}
    idx1, idx2 = random.randrange(len(h1)), random.randrange(len(h2)) # Índices al azar
    h1[idx1], h2[idx2] = h2[idx2], h1[idx1] # Intercambio directo de tareas
    return h1, h2, {'op': 'swap'}

def op_inter_shift(r1, r2):
    """Shift: Mueve una tarea de una ruta a otra (reubicación)."""
    h1, h2 = r1.copy(), r2.copy()
    if not h1: return h1, h2, {'info': 'vacío'}
    tarea = h1.pop(random.randrange(len(h1))) # Extrae de la ruta origen
    h2.insert(random.randint(0, len(h2)), tarea) # Inserta en la ruta destino
    return h1, h2, {'op': 'shift'}

def op_intra_mutacion(ruta):
    """2-Opt Intra-ruta: Invierte un segmento para optimizar el orden local."""
    h = ruta.copy()
    if len(h) < 2: return h, {'info': 'corta'}
    i, j = sorted(random.sample(range(len(h)), 2)) # Dos puntos de corte
    h[i:j+1] = reversed(h[i:j+1]) # Inversión del segmento intermedio
    return h, {'op': '2opt_intra'}

def aplicar_operador(tipo, padres):
    """Despachador con jerarquía optimizada para instancias restrictivas."""
    if tipo == "mutacion":
        hijo, meta = op_intra_mutacion(padres[0]) # Aplicamos inversión simple
        return [hijo], "intra_2opt", [meta]

    elif tipo == "cruce":
        rand = random.random() # Lanzamos moneda para elegir variedad
        # En instancias pequeñas, el SWAP (0.60 prob) es el que encuentra el BKS
        if rand < 0.60: 
            h1, h2, meta = op_inter_swap(padres[0], padres[1])
            nombre = "inter_swap"
        elif rand < 0.85: # SHIFT (25% prob) para balancear cargas
            h1, h2, meta = op_inter_shift(padres[0], padres[1])
            nombre = "inter_shift"
        else: # 2-OPT INTER (15% prob) para cambios estructurales
            h1, h2, meta = op_inter_2opt(padres[0], padres[1])
            nombre = "inter_2opt"
        return [h1, h2], nombre, [meta]
    return padres, "no_op", [{}]

In [81]:
c_

[6002, 9737, 5878, 5316]

In [82]:
cts_

26933

In [83]:
print(tf)

EVALUACIÓN COMPACTA DE RUTAS
RUTA 1 ['TR6', 'TR3', 'TR8', 'TR9']
  -> TR6 (2,6) -> DH: [1 -> 3 -> 2] | Demanda: 60 | Costo (DH + Serv): 1247 + 780 = 2027
  -> TR3 (2,3) -> DH: [6 -> 2] | Demanda: 25 | Costo (DH + Serv): 780 + 510 = 1290
  -> TR8 (3,4) -> DH: [Ninguno (ya en 3)] | Demanda: 7 | Costo (DH + Serv): 0 + 663 = 663
  -> TR9 (3,7) -> DH: [4 -> 3] | Demanda: 50 | Costo (DH + Serv): 663 + 311 = 974
  -> REGRESO A DEPÓSITO (1) -> DH: [7 -> 3 -> 1] | Costo Regreso: 1048
  => TOTAL RUTA 1: Costo Total = 6002 | Demanda Total = 142 / 150 [OK]

RUTA 2 ['TR7', 'TR12', 'TR5', 'TR2', 'TR10']
  -> TR7 (2,7) -> DH: [1 -> 3 -> 2] | Demanda: 30 | Costo (DH + Serv): 1247 + 370 = 1617
  -> TR12 (4,8) -> DH: [7 -> 4] | Demanda: 55 | Costo (DH + Serv): 468 + 1046 = 1514
  -> TR5 (2,5) -> DH: [8 -> 5 -> 2] | Demanda: 5 | Costo (DH + Serv): 984 + 745 = 1729
  -> TR2 (1,8) -> DH: [5 -> 8 -> 1] | Demanda: 27 | Costo (DH + Serv): 1321 + 1082 = 2403
  -> TR10 (3,8) -> DH: [8 -> 3] | Demanda: 23 | Cost

In [15]:
import math

# ==========================================
# 5. METAHEURÍSTICAS (ALTA VELOCIDAD)
# ==========================================

def calcular_costo_rapido(solucion, data, matriz_distancias):
    """Calcula el costo total de una solución en O(1) usando la matriz precalculada."""
    deposito = data.get('DEPOSITO', 1)
    info_tareas = {t['tarea']: t for t in data.get('LISTA_ARISTAS_REQ', [])}
    costo_total = 0
    
    for ruta in solucion:
        if not ruta: continue
        nodo_actual = deposito
        for id_tarea in ruta:
            tarea = info_tareas[id_tarea]
            u, v = tarea['nodos']
            
            costo_total += matriz_distancias[nodo_actual][u] + tarea['costo']
            nodo_actual = v
        costo_total += matriz_distancias[nodo_actual][deposito]
        
    return costo_total

In [85]:
#calcular_costo_rapido(new_sol, d_noreq, matriz_distancias_t)

In [86]:
calcular_costo_rapido(random_sol_t, d_noreq, matriz_distancias_t)

26933

In [16]:
## BKS 
import pandas as pd

In [17]:
benchmark = pd.read_csv(r'C:\Users\alhel\Desktop/TESIS CARP 2025/Instancias/Benchmarks.csv')
benchmark.head()

,Instances,BKS,BLB,BUB
0,kshs1,"14,661","14,661","14,661"
1,kshs2,"9,863","9,863","9,863"
2,kshs3,"9,320","9,320","9,320"
3,kshs4,"11,498","11,498","11,498"
4,kshs5,"10,957","10,957","10,957"


In [18]:
import pandas as pd
import numpy as np

def crear_diccionario_benchmarks(df_benchmarks):
    """
    Convierte un DataFrame de benchmarks en un diccionario de consulta rápida O(1).
    Estructura resultante: {'nombre_instancia': {'BKS': 100.0, 'BLB': 95.0, 'BUB': 105.0}}
    """
    diccionario_bks = {}
    
    if df_benchmarks is None or df_benchmarks.empty:
        return diccionario_bks
        
    # Función interna de limpieza de números
    def limpiar_numero(val):
        if pd.isna(val):
            return None
        try:
            if isinstance(val, str):
                val = val.replace(',', '').strip()
            return float(val)
        except (ValueError, TypeError):
            return None

    # Iteramos sobre el DataFrame y poblamos el diccionario
    for _, row in df_benchmarks.iterrows():
        # Estandarizamos el nombre de la llave (minúsculas y sin espacios a los lados)
        if 'Instances' not in row:
            continue
            
        nombre_instancia = str(row['Instances']).strip().lower()
        
        diccionario_bks[nombre_instancia] = {
            'BKS': limpiar_numero(row.get('BKS')),
            'BLB': limpiar_numero(row.get('BLB')),
            'BUB': limpiar_numero(row.get('BUB'))
        }
        
    return diccionario_bks

In [19]:
benchmark_dict = crear_diccionario_benchmarks(benchmark)
benchmark_dict

{'kshs1': {'BKS': 14661.0, 'BLB': 14661.0, 'BUB': 14661.0},
 'kshs2': {'BKS': 9863.0, 'BLB': 9863.0, 'BUB': 9863.0},
 'kshs3': {'BKS': 9320.0, 'BLB': 9320.0, 'BUB': 9320.0},
 'kshs4': {'BKS': 11498.0, 'BLB': 11498.0, 'BUB': 11498.0},
 'kshs5': {'BKS': 10957.0, 'BLB': 10957.0, 'BUB': 10957.0},
 'kshs6': {'BKS': 10197.0, 'BLB': 10197.0, 'BUB': 10197.0},
 'gdb1': {'BKS': 316.0, 'BLB': 316.0, 'BUB': 316.0},
 'gdb2': {'BKS': 339.0, 'BLB': 339.0, 'BUB': 339.0},
 'gdb3': {'BKS': 275.0, 'BLB': 275.0, 'BUB': 275.0},
 'gdb4': {'BKS': 287.0, 'BLB': 287.0, 'BUB': 287.0},
 'gdb5': {'BKS': 377.0, 'BLB': 377.0, 'BUB': 377.0},
 'gdb6': {'BKS': 298.0, 'BLB': 298.0, 'BUB': 298.0},
 'gdb7': {'BKS': 325.0, 'BLB': 325.0, 'BUB': 325.0},
 'gdb8': {'BKS': 348.0, 'BLB': 348.0, 'BUB': 348.0},
 'gdb9': {'BKS': 303.0, 'BLB': 303.0, 'BUB': 303.0},
 'gdb10': {'BKS': 275.0, 'BLB': 275.0, 'BUB': 275.0},
 'gdb11': {'BKS': 395.0, 'BLB': 395.0, 'BUB': 395.0},
 'gdb12': {'BKS': 458.0, 'BLB': 458.0, 'BUB': 458.0},
 'gdb13

In [20]:
import pandas as pd

def evaluar_gap_benchmarks(nombre_instancia, costo_actual, diccionario_bks):
    """
    Busca la instancia en el diccionario global y calcula los GAPs disponibles.
    Retorna un DataFrame de Pandas con el resumen de la métrica.
    """
    # Inicializamos valores nulos por defecto en caso de que falten datos
    bks, blb, bub = None, None, None
    gap_bks, gap_blb, gap_bub = None, None, None
    
    # Si tenemos los datos mínimos para buscar
    if diccionario_bks and nombre_instancia:
        nombre_limpio = str(nombre_instancia).strip().lower()
        limites = diccionario_bks.get(nombre_limpio)
        
        if limites:
            bks = limites.get('BKS')
            blb = limites.get('BLB')
            bub = limites.get('BUB')
            
            # Función lambda segura para calcular el porcentaje
            calc_pct = lambda base: ((costo_actual - base) / base) * 100 if base and base > 0 and costo_actual is not None else None

            if bks is not None: gap_bks = calc_pct(bks)
            if blb is not None: gap_blb = calc_pct(blb)
            if bub is not None: gap_bub = calc_pct(bub)

    # Creamos el diccionario estructurado para Pandas
    data = {
        "Instancia": [nombre_instancia],
        "BKS": [bks],
        "BLB": [blb],
        "BUB": [bub],
     #   "costo_solucion": [costo_actual],
        "GAP%": [gap_bks],
        "GAP_BLB%": [gap_blb],
        "GAP_BUB%": [gap_bub]
    }
    
    # Retornamos directamente el DataFrame
    return pd.DataFrame(data)

In [154]:
evaluar_gap_benchmarks('egl-s3-A', 5100, benchmark_dict)

,Instancia,BKS,BLB,BUB,GAP%,GAP_BLB%,GAP_BUB%
0,egl-s3-A,None,10165.0,10220.0,None,-49.827841,-50.097847


In [93]:
### RUN FUNCTION

In [21]:
def calcular_parametros_dinamicos(instance_d, costo_inicial):
    """
    Calcula los parámetros del Recocido Simulado de forma adaptativa 
    basándose en la topología de la instancia y la calidad de la solución inicial.
    """
    # 1. Extracción de características de la instancia
    # Ajusta 'ARISTAS_REQUERIDAS' al nombre exacto de la llave en tu diccionario
    num_tareas = instance_d.get('ARISTAS_REQUERIDAS', 0) 
    
    # Si tu diccionario guarda las tareas en una lista, la calculamos así:
    if num_tareas == 0 and 'ARISTAS' in instance_d:
        num_tareas = len([a for a in instance_d['ARISTAS'] if a.get('demanda', 0) > 0])
        
    # 2. Iteraciones por Temperatura (iter_por_t)
    # Regla: A mayor espacio de búsqueda, más iteraciones para alcanzar el equilibrio.
    # Usamos un multiplicador de 10x a 15x el número de tareas.
    iter_por_t = int(num_tareas * 15)
    
    # Ponemos límites de seguridad para no explotar la CPU
    iter_por_t = max(iter_por_t, 200)   # Mínimo vital para instancias enanas
    
    # 3. Tasa de Enfriamiento (alfa)
    # Instancias grandes necesitan enfriar más lento para no quedar atrapadas.
    alfa = 0.95  # Enfriamiento rápido para instancias fáciles

    # 4. Temperatura Inicial (t_inicial)
    # Criterio de Metrópolis: Queremos aceptar soluciones peores en un 10-20% 
    # al principio del algoritmo. Usamos el costo inicial como ancla.
    # Si la solución inicial cuesta 50,000, T0 será 5,000.
    t_inicial = costo_inicial * 0.10

    # 5. Probabilidad Inter-Ruta (p_inter)
    # En instancias masivas, cruzar camiones constantemente rompe la capacidad muy rápido.
    p_inter = 0.5  # Más agresivo (favorece intercambio entre camiones)

    # 6. Temperatura Final (t_final)
    # Valor de consenso universal
    t_final = 0.01

    return {
        "t_inicial": t_inicial,
        "alfa": alfa,
        "iter_por_t": iter_por_t,
        "t_final": t_final,
        "p_inter": p_inter
    }

# GENERAR VECINOS

In [22]:
import random
import copy

def aplicar_y_evaluar_vecindario(solucion, data, G, matriz_distancias, p_inter=0.5, max_intentos=100, generar_reporte=True):
    """
    Genera un vecino aplicando operadores a TODAS las rutas de la solución.
    Retorna: nueva_solucion, reporte_o_vacio, lista_operadores_usados
    """
    intentos = 0
    vecino_encontrado = False
    nueva = []
    detalles_cambio = ""
    operadores_aplicados = [] # <--- LISTA PARA GUARDAR LOS NOMBRES DE LOS OPERADORES
    
    while intentos < max_intentos:
        intentos += 1
        nueva = [ruta[:] for ruta in solucion]
        operadores_aplicados = [] # Reiniciamos en cada intento de la macro-perturbación
        
        rutas_con_tareas = [i for i, r in enumerate(nueva) if len(r) > 0]
        
        if not rutas_con_tareas:
            if generar_reporte: detalles_cambio = "No hay tareas en la solución para mover."
            break
            
        rutas_procesadas = set()
        rutas_a_procesar = list(rutas_con_tareas)
        random.shuffle(rutas_a_procesar)
        
        cambios_iteracion = []
        
        for r1 in rutas_a_procesar:
            if r1 in rutas_procesadas:
                continue
                
            es_inter = (random.random() < p_inter)
            
            # --- INTENTO DE CRUCE ---
            if es_inter:
                tipo = "cruce"
                rutas_disponibles = [i for i in rutas_con_tareas if i != r1 and i not in rutas_procesadas]
                
                if rutas_disponibles:
                    r2 = random.choice(rutas_disponibles)
                    padres = [nueva[r1], nueva[r2]]
                    hijos, nombre_op, metadata = aplicar_operador(tipo, padres)
                    
                    nueva[r1] = hijos[0]
                    nueva[r2] = hijos[1]
                    
                    rutas_procesadas.add(r1)
                    rutas_procesadas.add(r2)
                    
                    # GUARDAMOS EL NOMBRE DEL OPERADOR
                    operadores_aplicados.append(nombre_op)
                    
                    if generar_reporte:
                        estado_r2 = "VACIADA" if len(hijos[1]) == 0 else "ACTIVA"
                        cambios_iteracion.append(f"  -> CRUCE ({nombre_op}) | R{r1+1} y R{r2+1} | Meta: {metadata[0]}")
                else:
                    es_inter = False 
            
            # --- INTENTO DE MUTACIÓN ---
            if not es_inter:
                tipo = "mutacion"
                padres = [nueva[r1]]
                hijos, nombre_op, metadata = aplicar_operador(tipo, padres)
                
                nueva[r1] = hijos[0]
                rutas_procesadas.add(r1)
                
                # GUARDAMOS EL NOMBRE DEL OPERADOR
                operadores_aplicados.append(nombre_op)
                
                if generar_reporte:
                    cambios_iteracion.append(f"  -> MUTACIÓN ({nombre_op}) | R{r1+1} | Meta: {metadata[0]}")
        
        # --- VERIFICACIÓN DE FACTIBILIDAD ---
        if es_solucion_factible(nueva, data, matriz_distancias):
            vecino_encontrado = True
            if generar_reporte:
                detalles_cambio = f"MACRO-PERTURBACIÓN APLICADA (Intento {intentos}):\n" + "\n".join(cambios_iteracion)
            break 

    # ==========================================
    # PREPARACIÓN DE LA SALIDA
    # ==========================================
    
    # Si no se encontró vecino factible, la lista de operadores debe ser vacía o indicar fallo
    if not vecino_encontrado:
        operadores_aplicados = [] 

    if not generar_reporte:
        return (nueva if vecino_encontrado else solucion), "", operadores_aplicados

    # REPORTE DETALLADO (MODO DEBUG)
    reporte_debug = "\n" + "*"*80 + "\n🔍 DEBUG: APLICADOR DE MACRO-VECINDARIOS\n" + "*"*80 + "\n"
    
    if not vecino_encontrado:
        reporte_debug += f"\nOPERACIÓN FALLIDA:\n  -> Tras {max_intentos} intentos, no hubo factibilidad.\n"
        return solucion, reporte_debug, []
        
    reporte_debug += f"\nDETALLE DEL CAMBIO:\n{detalles_cambio}\n" + "*"*80 + "\n"
    
    return nueva, reporte_debug, operadores_aplicados

In [124]:
new_sol, report_neigh, op = aplicar_y_evaluar_vecindario(random_sol_t, d_noreq, G_test, matriz_distancias_t, p_inter=0.5, max_intentos=100, 
                                                     generar_reporte=True)

In [125]:
op

['inter_swap', 'intra_2opt', 'intra_2opt']

In [126]:
print(report_neigh)


********************************************************************************
🔍 DEBUG: APLICADOR DE MACRO-VECINDARIOS
********************************************************************************

DETALLE DEL CAMBIO:
MACRO-PERTURBACIÓN APLICADA (Intento 2):
  -> CRUCE (inter_swap) | R1 y R4 | Meta: {'op': 'swap'}
  -> MUTACIÓN (intra_2opt) | R3 | Meta: {'op': '2opt_intra'}
  -> MUTACIÓN (intra_2opt) | R2 | Meta: {'op': '2opt_intra'}
********************************************************************************



In [127]:
############ SA

In [23]:
import time
import math
import random
import pandas as pd

def sa_optimizar(nombre_instancia, data, G, matriz_distancias, dict_bks, alfa, p_inter, t_inicial = None, iter_por_t = None ):
    """
    Ejecuta el SA y retorna un DataFrame con:
    Métricas, BKS, GAP, Tiempo, Parámetros de configuración y Frecuencia de operadores.
    """
    # 0. GENERACIÓN DE SOLUCIÓN INICIAL
    solucion_inicial = generar_solucion_inicial_aleatoria(
        data=data, 
        matriz_distancias=matriz_distancias,
        nombre_instancia=nombre_instancia,  max_intentos=25
    )
    
    if not solucion_inicial:
        return pd.DataFrame([{"Instancia": nombre_instancia, "Error": "Fallo inicialización"}])

    # 1. INICIO CRONÓMETRO Y ESTADO
    t_inicio_cpu = time.process_time()
    solucion_actual = [r[:] for r in solucion_inicial]
    costo_actual = calcular_costo_rapido(solucion_actual, data, matriz_distancias)
    mejor_solucion, mejor_costo = [r[:] for r in solucion_actual], costo_actual

    # Diccionario para rastrear cuántas veces cada operador generó un vecino ACEPTADO
    frecuencias = {}

    # 2. CALIBRACIÓN DINÁMICA
    # Extraemos los parámetros para usarlos en el bucle y para el reporte final
    param_d = calcular_parametros_dinamicos(data, costo_actual)

    if not t_inicial:
        t_inicial = param_d["t_inicial"]
    if not iter_por_t:
        iter_por_t = param_d["iter_por_t"]

    t_final = param_d["t_final"]
   # p_inter = param_d["p_inter"]
    
    T = t_inicial
    
    # 3. BUCLE DE ENFRIAMIENTO
    while T > t_final:
        for _ in range(iter_por_t):
            # Recibimos: vecino, reporte (vacío aquí) y lista de operadores usados
            vecino, _, ops_usados = aplicar_y_evaluar_vecindario(
                solucion_actual, data, G, matriz_distancias, 
                p_inter=p_inter, max_intentos=10, generar_reporte=False
            )
            
            if vecino == solucion_actual:
                continue 
                
            costo_vecino = calcular_costo_rapido(vecino, data, matriz_distancias)
            delta = costo_vecino - costo_actual
            
            # 4. CRITERIO DE ACEPTACIÓN
            if delta <= 0 or random.random() < math.exp(-delta / T):
                solucion_actual = [r[:] for r in vecino]
                costo_actual = costo_vecino
                
                # Registramos las frecuencias de los operadores que compusieron este movimiento
                for op in ops_usados:
                    frecuencias[op] = frecuencias.get(op, 0) + 1
                
                if costo_actual < mejor_costo:
                    mejor_solucion = [r[:] for r in solucion_actual]
                    mejor_costo = costo_actual
                    
        T *= alfa
        
    tiempo_cpu = time.process_time() - t_inicio_cpu
    
    # 5. EXTRACCIÓN DE BKS Y CÁLCULO DE GAP
    nombre_instancia_clean = str(nombre_instancia).strip().lower()
    bks_valor = dict_bks.get(nombre_instancia_clean)
    df_gaps = evaluar_gap_benchmarks(nombre_instancia_clean, mejor_costo, dict_bks)
  #  gap_final = df_gaps["gap%"].iloc[0] if not df_gaps.empty else None

    # ==========================================
    # 6. CONSTRUCCIÓN DEL DATAFRAME DE RESULTADOS
    # ==========================================
    # 1. Datos de Desempeño
    resultado_dict = {
        "Instancia": nombre_instancia,
      #  "BKS": bks_valor,
        "Mejor_Costo": mejor_costo,
      #  "GAP_%": gap_final,
        "Tiempo_CPU_seg": round(tiempo_cpu, 4),
    }

    # 2. Parámetros Utilizados (Metadatos de la corrida)
    resultado_dict.update({
        "Param_T0": round(t_inicial, 2),
        "Param_Alpha": alfa,
        "Param_Iter_x_T": iter_por_t,
        "Param_T_Final": t_final,
        "Param_P_Inter": p_inter
    })
    
    # 3. Frecuencias de Operadores
    resultado_dict.update(frecuencias)
    
    # 4. Solución final (al final por ser un string largo)
    resultado_dict["Mejor_Solucion"] = str(mejor_solucion)
    
    # Crear DataFrame de una sola fila
    df_resultado = pd.DataFrame([resultado_dict])
    
    # Llenar con 0 los operadores que no se usaron en esta instancia
    df_resultado = df_resultado.merge(df_gaps, on = "Instancia")#.fillna(0)
        
    return df_resultado

In [159]:
sa_optimizar(
    nombre_instancia=d_noreq['NOMBRE'], 
    data=d_noreq, 
    G=G_test, 
    matriz_distancias=matriz_distancias_t, 
    dict_bks=benchmark_dict, alfa = 0.9, p_inter = 0.7
)

,Instancia,Mejor_Costo,Tiempo_CPU_seg,Param_T0,Param_Alpha,Param_Iter_x_T,Param_T_Final,Param_P_Inter,inter_swap,inter_2opt,intra_2opt,inter_shift,Mejor_Solucion,BKS,BLB,BUB,GAP%,GAP_BLB%,GAP_BUB%
0,kshs1,17930,4.625,2585.5,0.9,200,0.01,0.7,1308,253,2598,242,"[['TR6', 'TR14', 'TR10'], ['TR1', 'TR9', 'TR3'...",14661.0,14661.0,14661.0,22.297251,22.297251,22.297251


In [248]:
##### loop

In [24]:
import os
import pandas as pd
import time

def basic_run(instance_path, run_id, benchmark_dict, alpha, p_im, t_ini, iterac_ , algoritmo_dist='dijkstra'):
    """
    Orquestador: Lee, optimiza y guarda el CSV individual en la carpeta del experimento.
    """
    # 1. Lectura de la instancia
    instance_d, _ = leer_carplib_dat(instance_path)
    val_d, _ = validate_instance(instance_d, instance_path)
    
    if not val_d['is_valid']:
        print(f"❌ Error: Falló lectura en {instance_path}")
        return None

    instance_name = instance_d['NOMBRE']

    # 2. Definir y Crear Carpeta de Guardado
    # Ruta: EXPERIMENTOS / [run_id]
    path_experimento = os.path.join("EXPERIMENTOS", run_id)
    os.makedirs(path_experimento, exist_ok=True) # Crea la carpeta si no existe

    # 3. Procesamiento (Grafo y Distancias)
    g_object = generar_grafo_limpio(instance_d, carpeta_salida=None, figsize=(35,30), k_layout=15)
    matriz_caminos = calcular_matriz_distancias(g_object, algoritmo=algoritmo_dist, 
                                               carpeta_salida=None, nombre_instancia=instance_name)

    # 4. Optimización SA
    df_experimento = sa_optimizar(
        nombre_instancia=instance_name,
        data=instance_d,
        G=g_object,
        matriz_distancias=matriz_caminos,
        dict_bks=benchmark_dict, alfa = alpha, 
        p_inter = p_im, t_inicial = t_ini, iter_por_t = iterac_ )
    
    # 5. Guardado Individual
    if df_experimento is not None and not df_experimento.empty:
        df_experimento.insert(0, "Run_ID", run_id)
        df_experimento.insert(1, "Algoritmo_Distancias", algoritmo_dist)
        
        # Nombre: Instancia_RUN_ID_XXX.csv
        nombre_csv = f"{instance_name}_RUN_ID_{run_id}.csv"
        ruta_csv = os.path.join(path_experimento, nombre_csv)
        
        df_experimento.to_csv(ruta_csv, index=False)
        print(f"📊 Guardado individual: {nombre_csv}")
    
    return df_experimento

def ejecutar_experimento_completo(folder_raiz, run_id_global, benchmark_dict, a, p, t_, it_ , algoritmo_dist='dijkstra'):
    """
    Recorre subcarpetas, ejecuta las instancias y genera el reporte maestro 
    dentro de la carpeta específica del experimento.
    """
    lista_resultados = []
    
    # Asegurar que la carpeta base EXPERIMENTOS existe
    os.makedirs("EXPERIMENTOS", exist_ok=True)
    
    # 1. Búsqueda de archivos
    rutas_archivos = []
    for root, dirs, files in os.walk(folder_raiz):
        for file in files:
            if file.endswith('.dat'):
                rutas_archivos.append(os.path.join(root, file))
    
    if not rutas_archivos:
        print(f"⚠️ No hay archivos .dat en {folder_raiz}")
        return None

    print(f"🚀 Iniciando Run: {run_id_global} sobre {len(rutas_archivos)} instancias.")

    # 2. Iteración sobre instancias
    for i, ruta_completa in enumerate(rutas_archivos):
        nombre_archivo = os.path.basename(ruta_completa)
        print(f"\n--- [{i+1}/{len(rutas_archivos)}] Procesando: {nombre_archivo}")
        
        try:
            # Enviamos el run_id_global para que todas las instancias se guarden juntas
            df_instancia = basic_run(
                instance_path=ruta_completa, 
                run_id=run_id_global, 
                benchmark_dict=benchmark_dict, 
                algoritmo_dist=algoritmo_dist, 
                alpha = a , 
                p_im = p, 
                t_ini = t_, 
                iterac_ = it_
            )
            if df_instancia is not None and not df_instancia.empty:
                df_instancia["Ruta_Origen"] = os.path.dirname(ruta_completa)
                lista_resultados.append(df_instancia)
                
        except Exception as e:
            print(f"❌ Error en {nombre_archivo}: {str(e)}")
            continue

    # 3. Consolidación Final
    if lista_resultados:
        df_maestro = pd.concat(lista_resultados, ignore_index=True).fillna(0)
        
        # El reporte maestro se guarda dentro de la carpeta del run_id
        ruta_maestro = os.path.join("EXPERIMENTOS", run_id_global, "REPORTE_MAESTRO.csv")
        df_maestro.to_csv(ruta_maestro, index=False)
        
        print("\n" + "="*60)
        print(f"✅ EXPERIMENTO {run_id_global} FINALIZADO")
        print(f"📄 Reporte Maestro: {ruta_maestro}")
        print("="*60)
        
        return df_maestro
    return None

# Ejemplo de ejecución:
df_final = ejecutar_experimento_completo("Instancias", "EXP_001", benchmark_dict, 0.9, 0.7, None, None)

🚀 Iniciando Run: EXP_001 sobre 87 instancias.

--- [1/87] Procesando: 10A.dat

--- [2/87] Procesando: 10B.dat

--- [3/87] Procesando: 10C.dat

--- [4/87] Procesando: 10D.dat

--- [5/87] Procesando: 1A.dat

--- [6/87] Procesando: 1B.dat

--- [7/87] Procesando: 1C.dat
❌ Error en 1C.dat: No se halló solución factible tras 25 intentos.

--- [8/87] Procesando: 2A.dat

--- [9/87] Procesando: 2B.dat

--- [10/87] Procesando: 2C.dat
❌ Error en 2C.dat: No se halló solución factible tras 25 intentos.

--- [11/87] Procesando: 3A.dat

--- [12/87] Procesando: 3B.dat

--- [13/87] Procesando: 3C.dat
❌ Error en 3C.dat: No se halló solución factible tras 25 intentos.

--- [14/87] Procesando: 4A.dat

--- [15/87] Procesando: 4B.dat

--- [16/87] Procesando: 4C.dat

--- [17/87] Procesando: 4D.dat

--- [18/87] Procesando: 5A.dat

--- [19/87] Procesando: 5B.dat

--- [20/87] Procesando: 5C.dat

--- [21/87] Procesando: 5D.dat

--- [22/87] Procesando: 6A.dat

--- [23/87] Procesando: 6B.dat

--- [24/87] Procesand


KeyboardInterrupt



In [ ]:
import os
import pandas as pd
import time

def grid_search_metaheuristica(folder_raiz, lista_alphas, lista_p_inters, replicas, benchmark_dict):
    """
    Ejecuta un Grid Search variando Alpha y P_Inter.
    Crea una carpeta EXP_XXX para cada combinación y réplica.
    """
    contador_exp = 1
    reporte_resumen_grid = []

    print(f"🔬 Iniciando Grid Search: {len(lista_alphas)} alphas x {len(lista_p_inters)} p_inters x {replicas} réplicas")
    print(f"📁 Total de experimentos a ejecutar: {len(lista_alphas) * len(lista_p_inters) * replicas}")

    for alpha in lista_alphas:
        for p_inter in lista_p_inters:
            for r in range(1, replicas + 1):
                
                # Generamos el run_id con formato EXP_001, EXP_002...
                run_id_global = f"EXP_{contador_exp:03d}"
                
                print(f"\n{'='*70}")
                print(f"🚀 EJECUTANDO: {run_id_global} | Alpha: {alpha} | P_Inter: {p_inter} | Réplica: {r}/{replicas}")
                print({'='*70})

                # Llamamos a tu función de experimento completo pasándole los parámetros fijos
                # Nota: t_initial e iteraciones se pasan como None para que se calculen dinámicamente
                df_maestro_run = ejecutar_experimento_completo(
                    folder_raiz=folder_raiz,
                    run_id_global=run_id_global,
                    benchmark_dict=benchmark_dict,
                    alpha=alpha,
                    p_inter=p_inter,
                    t_initial=None, 
                    iteraciones=None,
                    algoritmo_dist='dijkstra'
                )

                if df_maestro_run is not None:
                    # Guardamos un resumen de esta corrida para el análisis final del Grid Search
                    resumen_run = {
                        "Run_ID": run_id_global,
                        "Alpha": alpha,
                        "P_Inter": p_inter,
                        "Replica": r,
                        "GAP_Promedio": df_maestro_run["GAP_%"].mean(),
                        "Tiempo_Total_seg": df_maestro_run["Tiempo_CPU_seg"].sum()
                    }
                    reporte_resumen_grid.append(resumen_run)
                
                contador_exp += 1

    # Al finalizar todo el Grid Search, guardamos un archivo de control de calidad
    df_grid_final = pd.DataFrame(reporte_resumen_grid)
    os.makedirs("EXPERIMENTOS", exist_ok=True)
    df_grid_final.to_csv("EXPERIMENTOS/RESUMEN_GRID_SEARCH.csv", index=False)
    
    print("\n" + "🌟"*20)
    print("GRID SEARCH FINALIZADO")
    print("Revisa EXPERIMENTOS/RESUMEN_GRID_SEARCH.csv para ver qué combinación ganó.")
    print("🌟"*20)

    return df_grid_final

# ==========================================
# EJEMPLO DE USO
# ==========================================
alphas_a_probar = [0.90, 0.95, 0.98]
pinters_a_probar = [0.4, 0.6, 0.8]
replicas_por_celda = 5

df_resultados_grid = grid_search_metaheuristica(
    "Instancias", 
     alphas_a_probar, 
     pinters_a_probar, 
     replicas_por_celda, 
     benchmark_dict
 )